In [1]:
!pip install optuna
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
device = ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on : {device}")

Running on : cuda


In [4]:
train_df = pd.read_csv("/kaggle/input/datasets/hamzakamal6969/fashion-mnist/fashion-mnist_train.csv")
test_df = pd.read_csv("/kaggle/input/datasets/hamzakamal6969/fashion-mnist/fashion-mnist_test.csv")
train_df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["label"],
    random_state=42
)

In [6]:
X_train = train_df.iloc[:,1:].values
X_test = test_df.iloc[:,1:].values
cv_features = val_df.iloc[:,1:].values
cv_labels = val_df.iloc[:,0].values
y_train = train_df.iloc[:,0].values
y_test =  test_df.iloc[:,0].values


In [7]:
import numpy as np

print(np.isnan(X_train).sum())
print(np.isnan(X_test).sum())

0
0


In [8]:
X_train = X_train/255
X_test = X_test/255
cv_features= cv_features/255
X_train.shape


(48000, 784)

In [9]:
class CustomDataset(Dataset) :

  def __init__(self,features,labels) :

    self.features = torch.tensor(features,dtype= torch.float32)
    self.labels = torch.tensor(labels,dtype= torch.long)

  def __len__(self) :

    return len(self.features)

  def __getitem__(self, index):

    return self.features[index], self.labels[index]


In [10]:
cv_dataset = CustomDataset(cv_features,cv_labels)

In [11]:
train_dataset = CustomDataset(X_train, y_train)

In [12]:
test_dataset = CustomDataset(X_test, y_test)

In [13]:
# create train and test loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [14]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, x):

    return self.model(x)

In [15]:
# objective function
def objective(trial):

  # next hyperparameter values from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 10)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
  epochs = trial.suggest_int("epochs", 10, 50, step=10)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  cv_loader = DataLoader(cv_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

  # model init
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # optimizer selection
  criterion = nn.CrossEntropyLoss()

  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop

  for epoch in range(epochs):

    for batch_features, batch_labels in train_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()


  # evaluation
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in cv_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

  return accuracy

In [16]:
import optuna

study = optuna.create_study(direction='maximize',sampler = optuna.samplers.TPESampler())
study.optimize(objective, n_trials=30)

[I 2026-10-16 15:48:53,165] A new study created in memory with name: no-name-47f02ba3-9f61-4560-bcd7-bbdb75600108
[I 2026-10-16 15:55:24,367] Trial 0 finished with value: 0.66625 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 64, 'epochs': 40, 'learning_rate': 0.03221013680027276, 'dropout_rate': 0.4, 'batch_size': 16, 'optimizer': 'Adam', 'weight_decay': 5.787948492667074e-05}. Best is trial 0 with value: 0.66625.
[I 2026-10-16 15:58:02,366] Trial 1 finished with value: 0.5640833333333334 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 56, 'epochs': 40, 'learning_rate': 0.0925743719842023, 'dropout_rate': 0.30000000000000004, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 7.44234338969095e-05}. Best is trial 0 with value: 0.66625.
[I 2026-10-16 15:59:50,213] Trial 2 finished with value: 0.6141666666666666 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 72, 'epochs': 20, 'learning_rate': 0.0012304574969706167, 'dropout_rate': 0.5, '

In [17]:
best_params = study.best_params

num_hidden_layers = best_params["num_hidden_layers"]
neurons_per_layer = best_params["neurons_per_layer"]
epochs = best_params["epochs"]
learning_rate = best_params["learning_rate"]
dropout_rate = best_params["dropout_rate"]
batch_size = best_params["batch_size"]
optimizer_name = best_params["optimizer"]
weight_decay = best_params["weight_decay"]

In [18]:
study.best_value


0.895

In [19]:
model = MyNN(784, 10, num_hidden_layers, neurons_per_layer, dropout_rate)
model= model.to(device)
criterion = nn.CrossEntropyLoss()

if optimizer_name == 'Adam':
  optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
elif optimizer_name == 'SGD':
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
else:
  optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)


In [20]:
for epoch in range(epochs):

    model.train()

    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [21]:
total = 0
correct = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

    accuracy = correct/total

    print(accuracy)

0.8747
